In [1]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList, TextIteratorStreamer
from threading import Thread
from transformers import GenerationConfig

from opencc import OpenCC
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList, TextIteratorStreamer
from threading import Thread
from transformers import GenerationConfig

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from opencc import OpenCC
import gradio as gr
from transformers.generation.stopping_criteria import StoppingCriteria, StoppingCriteriaList

s2t = OpenCC('s2t')  # 將簡體中文轉換為繁體中文
t2s = OpenCC('t2s')  # 將繁體中文轉換為簡體中文

CUDA_AVAILABLE = torch.cuda.is_available()
device = torch.device("cuda" if CUDA_AVAILABLE else "cpu")

model_name_or_path = './my-pretrained-3epochs-YeungNLP-zh-ch'

if CUDA_AVAILABLE:
    model = AutoModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype='auto', ignore_mismatched_sizes=True)
else:
    model = AutoModelForCausalLM.from_pretrained(model_name_or_path, ignore_mismatched_sizes=True)

tokenizer = AutoTokenizer.from_pretrained(
    'Langboat/bloom-389m-zh',
    #'YeungNLP/bloomz-396m-zh',
    trust_remote_code=True,
    use_fast=True
)

class StopOnTokens(StoppingCriteria):
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        stop_ids = [tokenizer.eos_token_id]
        for stop_id in stop_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

def predict(message, history):
    history_transformer_format = [[message, ""]]
    stop = StopOnTokens()

    messages = "".join(["".join(["<Human>: \n"+item[0], "\n\n<Assistant>:"+item[1]]) for item in history_transformer_format])
    messages = t2s.convert(messages)
    
    model_inputs = tokenizer([messages], return_tensors="pt", add_special_tokens=False).to(device)
    
    generate_kwargs = dict(
        model_inputs,
        max_length=400,
        do_sample=True,
        top_p=0.95,
        top_k=200,
        temperature=1.0,
        repetition_penalty=1.2,
        num_beams=1,
        stopping_criteria=StoppingCriteriaList([stop])
    )

    output = model.generate(**generate_kwargs)
    partial_message = s2t.convert(tokenizer.decode(output[0], skip_special_tokens=True))
    return partial_message

# Examples for Gradio interface
examples = [
    ["房間大小舒適"],
    ["房內有蚊子在飛"],
    ["才剛進房就聞到菸味，久久不散"],
    ["浴室環境能再改善"],
    ["隔音很差"],
    ["交通不便"],
    # ... (add more examples)
]

# Gradio ChatInterface setup
title = "自己微調的小型ChatGPT"
description = "微調GPT，讓它可以回答各式各樣的問題，就像人類對話一般"

chatinterface = gr.ChatInterface(fn=predict,
                                examples=examples,
                                title=title,
                                description=description,
                                textbox=gr.Textbox(value="", placeholder="Ask me a question", container=False, lines=1, scale=5),
                                theme="soft",
                                retry_btn="再產生一次答案",
                                undo_btn="刪除最後一次對談",
                                clear_btn="新的交談")               
chatinterface.queue()
chatinterface.launch()


Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.
